<div align="center">

# SHIV'S LLM ENGINEERING JOURNEY

### DAY 05 · Building a Business Solution

*Combining LLM calls to research a company website and generate a business brochure.*

**Learn · Build · Experiment · Share**

</div>

---

> **Learning in public** — a hands-on notebook documenting my progress in Generative AI and Agentic AI.

### What you'll explore

- Relevant-link selection
- Structured JSON responses
- Multi-step LLM workflows
- Business content generation

---

**Author:** Shiv Bhadouria  
**Focus:** Generative AI · LLM APIs · Prompt Engineering · Agentic AI



# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
import os
from openai import OpenAI

# Initialize environment variables
load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

# Validate Google API key
if google_api_key and len(google_api_key) > 10:
    print("Google API key looks good so far!")
else:
    print("There might be a problem with your Google API key. Please check your .env file!")

# Gemini configuration
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

MODEL = "gemini-3.1-flash-lite"

# Initialize Gemini client
gemini = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key
) if google_api_key else None

Google API key looks good so far!


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

### First step: Have Gemini identify which links are relevant
Use Gemini to analyze webpage links and return structured JSON.
We will use Gemini to examine the links available on a webpage and decide which ones are relevant to our objective.

The model should:
- Identify links that are relevant to the given objective.
- Ignore irrelevant links, such as privacy policies, cookie settings, and unrelated pages.
- Convert relative URLs such as /about into absolute URLs such as https://company.com/about.
- Return the results in a structured JSON format.
We will use one-shot prompting, where we provide Gemini with an example of the expected input and output in the prompt.

This is an excellent use case for an LLM because determining whether a link is relevant requires understanding its context and meaning, rather than simply matching keywords.

Sidenote: A more advanced technique is called Structured Outputs, where we define a schema that the model must follow when generating its response. This helps ensure consistent, machine-readable JSON output. We will explore this technique during Week 8 in our Autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [9]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.1-flash-lite
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum/experience page',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
13 days ago
•
146k
•
3.94k
Nanbeige/Nanbeige4.3-33B
Updated
2 days ago
•
12.2k
•
791
LiquidAI/LFM3-1.7B
Updated
2 days ago
•
23.7k
•
525
nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B
Updated
about 14 hours ago
•
16k
•
462
ISTA-DASLab/

In [26]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated\n13 days ago\n•\n146k\n•\n3.94k\nNanbeige/Nanbeige4.3-33B\nUpdated\n2 days 

In [20]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


# Hugging Face: The AI Community Building the Future

### Who We Are
Hugging Face is the global home of machine learning. We are a collaborative platform dedicated to democratizing artificial intelligence by providing the tools, infrastructure, and community necessary to build the future of technology. Whether you are an individual researcher, an open-source contributor, or a Fortune 500 enterprise, Hugging Face serves as the central hub for discovering and deploying state-of-the-art AI.

### Our Ecosystem
We provide an integrated environment where the machine learning community can collaborate on every stage of development:
* **Models:** Access over 2 million pre-trained machine learning models.
* **Datasets:** Explore half a million datasets to train and refine your AI projects.
* **Spaces:** Deploy and share interactive AI applications instantly.
* **Collaboration:** Join a vibrant community to discuss papers, share research, and contribute to the evolution of AI through our forums and discord.

### Empowering Enterprises
Hugging Face bridges the gap between research and production. Our enterprise solutions are designed to help organizations scale their AI initiatives securely and efficiently. With features like **Inference Endpoints**, **Enterprise Support**, and **Storage Buckets**, we provide the professional-grade infrastructure needed to turn cutting-edge models into real-world business value.

### Company Culture
Hugging Face is built on the principles of **Open Source** and **Collaboration**. Our culture is defined by transparency and knowledge sharing. We are a community-first organization, actively pushing the boundaries of what is possible in AI by fostering an ecosystem where the best ideas—from deep-learning papers to innovative agents—are accessible to everyone. We believe that by working together, the community can solve the most challenging problems in machine learning.

### Careers: Join the Team
We are looking for passionate individuals who want to help us shape the future of AI. Whether you are an engineer, a researcher, or a community builder, joining Hugging Face means working at the epicenter of the AI revolution. 

As a member of our team, you will:
* Work on the world's most influential ML platform.
* Engage directly with the open-source community.
* Solve complex problems at the intersection of infrastructure, research, and application.

**Ready to build the future with us?** 
Explore our platform at [huggingface.co](https://huggingface.co) to see what the community is creating, or reach out to us to learn more about how our enterprise tools can accelerate your machine learning roadmap.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from Gemini,
with the familiar typewriter animation

In [28]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        reasoning_effort="none",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [29]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


# Welcome to Hugging Face: Where the Robots Come to Play

So, you’ve heard about the AI revolution? While everyone else is busy debating whether the machines will take over or just turn us into paperclips, we’ve decided to invite them over for coffee. 

Welcome to **Hugging Face**, the global clubhouse for the machine learning community. Think of us as the digital equivalent of a massive, caffeinated workshop where 2 million+ models, half a million datasets, and a gazillion lines of code live in perfect (mostly) harmony.

---

### What’s the Vibe?
We aren’t just a company; we’re an open-source movement with a very cute emoji for a logo. 

* **Our Culture:** We believe that building the future of AI shouldn’t be a lonely endeavor inside a dark server room. We’re all about collaboration, sharing, and obsessing over the latest research papers until 3:00 AM. If you enjoy solving problems, breaking things to see how they work, and talking shop about "Agentic RL" or "Tokenmaxxing," you’ll fit right in.
* **Our Playground:** With over 2 million models and 1 million applications (Spaces), our platform is where the magic happens—from video generators that can swap your face with a medieval portrait to AI agents that actually do your chores (well, your coding chores, at least).

---

### Who Hangs Out Here?
Basically, everyone who is anyone in AI.
* **The Researchers:** The folks pushing the boundaries of what’s possible with foundation models.
* **The Developers:** The legends building the next generation of agents, bots, and tools.
* **The Enterprise Giants:** Companies looking to tap into the open-source ecosystem without having to reinvent the wheel every Tuesday.

---

### Want to Join the Chaos?
Are you a developer, a researcher, or just someone who thinks "Hugging Face" is the best name for a tech company ever? (We agree). 

We are constantly looking for curious minds to help us build the infrastructure of the future. Whether you want to contribute to our open-source libraries, help enterprises integrate our tech, or dive deep into the weeds of hardware optimization, there is a seat at the table.

**Why work here?**
1. **Be at the Center:** You’ll be working at the absolute heartbeat of the global AI ecosystem.
2. **Impact:** Your work isn't hidden in a silo; it’s used by the entire world.
3. **The Culture:** We take our work seriously, but we don't take *ourselves* seriously. (Seriously, have you seen our mascot?)

---

### Ready to get started?
Stop staring at the screen and jump into the Hub. Browse the trending models, try out a Space, or dive into our documentation. 

**Hugging Face: The AI community building the future. Join us before the machines decide they don't need us anymore.** 

*P.S. Yes, the hug is mandatory. (Kidding... mostly.)*

In [30]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Welcome to Hugging Face: The Internet’s Favorite Emoji (and AI Powerhouse)

### So, Who Are We?
Think of us as the town square for the Machine Learning world. If AI were a giant, galaxy-brain potluck dinner, Hugging Face is the table, the kitchen, and the extremely helpful person handing out recipes. We are the platform where the global AI community congregates to build, share, and inevitably argue about the future of technology.

### Why Do People Love Us?
We are the home of over 2 million models and half a million datasets. If you need a neural network that can generate a video of a cat playing jazz, or a dataset to train an agent to solve global crises, we’ve got it. We are essentially the "GitHub of AI," but with significantly more emojis and less existential dread.

### Our Vibe (Culture)
We take our technology seriously, but we don't take ourselves seriously. We believe the future of AI is open-source, collaborative, and powered by the collective intelligence of the planet. Whether you’re a researcher in a lab or a teenager building a chatbot in your bedroom, you have a seat at our table. Our culture is built on transparency, rapid-fire innovation, and a shared obsession with making machine learning accessible to everyone—not just the folks in ivory towers.

### Who Are Our Customers?
Everyone from individual developers building "Moon Bots" on their lunch break to massive enterprise players utilizing our Inference Endpoints. If you are serious about AI, you are already using us. We provide the infrastructure (the buckets, the hardware, the compute) so you can focus on the "magic" (the actual modeling).

### Career Opportunities: Why Join the Chaos?
Do you like being at the center of the AI revolution? Do you dream in Transformers and wake up to a fresh batch of papers? We are always looking for:
* **Engineers** who want to break things to see how they work.
* **Researchers** obsessed with the frontier of intelligence.
* **Community Builders** who can herd cats (or data scientists, which is harder).

If you want to contribute to the platform that powers the most advanced models in the world—and work for a company that colors its world in bright yellow and orange—you belong here.

### The TL;DR
* **Models:** 2 Million+.
* **Datasets:** 500,000+.
* **Spaces:** Infinite possibilities.
* **Our Mission:** To democratize good machine learning, one commit at a time.

**Come for the models, stay for the community, and help us build the future before the robots do it for us.**

*Visit us at huggingface.co—we’re the ones with the friendly face.*


<div style="padding:24px; border-radius:12px; background:#f8fafc; border:1px solid #e2e8f0;">

  <h2 style="margin-top:0; color:#166534;">
    Business Applications
  </h2>

  <p style="color:#334155; line-height:1.7;">
    In this exercise, we extended our Day 1 implementation by combining
    multiple LLM calls to generate a complete business document.
  </p>

  <p style="color:#334155; line-height:1.7;">
    This is an early example of <strong>Agentic AI</strong>, where multiple
    LLM interactions work together to accomplish a task. We will explore
    these concepts further in Week 2 and build a fully autonomous AI agent
    in Week 8.
  </p>

  <p style="color:#334155; line-height:1.7; margin-bottom:0;">
    <strong>Real-world applications:</strong> Marketing content, product
    tutorials, personalized emails, business reports, and automated
    document generation.
  </p>

</div>
